# Azure AI Speech: Text-to-Speech and Speech-to-Text

In this notebook you will learn how to use **Azure AI Speech** to:

1. Convert text into natural-sounding audio (**Text-to-Speech**)
2. Control pronunciation and pacing with **SSML** (Speech Synthesis Markup Language)
3. Transcribe an audio file back into text (**Speech-to-Text**)

---

### Prerequisites: Deploy the Azure Resource

Before running this notebook, you need an **Azure AI Speech** resource. Instead of configuring it manually in the Azure Portal, deploy the included **ARM template** which creates the resource in the correct configuration automatically.

> **Tip:** Open `deploy-speech.parameters.json` to customize the resource name, region, or SKU before deploying. The default SKU is **S0** (paid). Change it to **F0** for the free tier.

**1. Deploy the resource** — run the cell below (replace `<your-resource-group>`):

In [ ]:
import subprocess, shutil

# ⚠️ Replace <your-resource-group> with your actual resource group name
RESOURCE_GROUP = "Foundry"

# Resolve the full path to the Azure CLI (needed for Windows where az is a .cmd file)
AZ = shutil.which("az")
if not AZ:
    raise FileNotFoundError("Azure CLI (az) not found. Install it from https://learn.microsoft.com/cli/azure/install-azure-cli")

result = subprocess.run(
    [AZ, "deployment", "group", "create",
     "--resource-group", RESOURCE_GROUP,
     "--template-file", "deploy-speech.json",
     "--parameters", "deploy-speech.parameters.json"],
    capture_output=True, text=True, encoding="utf-8"
)
print(result.stdout)
if result.returncode != 0:
    print("ERROR:", result.stderr)

**2. Assign the required RBAC role** so your identity can use the resource (replace `<your-resource-group>`):

> **Note:** RBAC role assignments can take **1–2 minutes** to propagate. If you get a 401 error on your first run, wait a moment and try again.

In [ ]:
import subprocess, shutil

RESOURCE_GROUP = "Foundry"
AZ = shutil.which("az")
if not AZ:
    raise FileNotFoundError("Azure CLI (az) not found. Install it from https://learn.microsoft.com/cli/azure/install-azure-cli")

# Step A: Get the signed-in user's Object ID
user_result = subprocess.run(
    [AZ, "ad", "signed-in-user", "show", "--query", "id", "-o", "tsv"],
    capture_output=True, text=True, encoding="utf-8"
)
user_id = user_result.stdout.strip()
print(f"Your User ID: {user_id}")

# Step B: Get the resource ID from the deployment output
scope_result = subprocess.run(
    [AZ, "deployment", "group", "show",
     "--resource-group", RESOURCE_GROUP,
     "--name", "deploy-speech",
     "--query", "properties.outputs.resourceId.value", "-o", "tsv"],
    capture_output=True, text=True, encoding="utf-8"
)
resource_id = scope_result.stdout.strip()
print(f"Resource ID:  {resource_id}")

# Step C: Assign the RBAC role
role_result = subprocess.run(
    [AZ, "role", "assignment", "create",
     "--assignee", user_id,
     "--role", "Cognitive Services Speech User",
     "--scope", resource_id],
    capture_output=True, text=True, encoding="utf-8"
)
print(role_result.stdout)
if role_result.returncode != 0:
    print("ERROR:", role_result.stderr)

**3. Update the `.env` file** in this folder with the values from the deployment output:

```
SPEECH_ENDPOINT="https://<your-account-name>.cognitiveservices.azure.com/"
SPEECH_REGION="eastus2"
SPEECH_RESOURCE_ID="/subscriptions/<sub-id>/resourceGroups/<rg>/providers/Microsoft.CognitiveServices/accounts/<account-name>"
```

You can retrieve all three values at once by running the cell below (replace `<your-resource-group>`):

In [ ]:
import subprocess, shutil

RESOURCE_GROUP = "Foundry"
AZ = shutil.which("az")
if not AZ:
    raise FileNotFoundError("Azure CLI (az) not found. Install it from https://learn.microsoft.com/cli/azure/install-azure-cli")

result = subprocess.run(
    [AZ, "deployment", "group", "show",
     "--resource-group", RESOURCE_GROUP,
     "--name", "deploy-speech",
     "--query", "properties.outputs"],
    capture_output=True, text=True, encoding="utf-8"
)
print(result.stdout)
if result.returncode != 0:
    print("ERROR:", result.stderr)

### Authentication

This notebook uses **Microsoft Entra ID** (`DefaultAzureCredential`) instead of API keys.
Make sure you are logged in with `az login`.

> **Note:** The Speech SDK requires the auth token in a special format:
> `aad#<resource-id>#<token>`. This is handled automatically in Step 2.

## Step 1: Install the Speech SDK

In [ ]:
%pip install azure-cognitiveservices-speech azure-identity python-dotenv

## Step 2: Load Environment Variables and Obtain a Token

The Speech SDK doesn't accept `DefaultAzureCredential` directly. Instead
we request a token from Entra ID and pass it as an **authorization token**.

The Speech SDK requires the token in a special format: `aad#<resource-id>#<token>`.
This tells the SDK to use Entra ID authentication with the specified resource.

In [ ]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
import azure.cognitiveservices.speech as speechsdk

load_dotenv(override=True)

speech_endpoint = os.getenv("SPEECH_ENDPOINT")
speech_region = os.getenv("SPEECH_REGION")
speech_resource_id = os.getenv("SPEECH_RESOURCE_ID")

# Obtain a token from Entra ID
credential = DefaultAzureCredential()
token = credential.get_token("https://cognitiveservices.azure.com/.default").token

# Build the SpeechConfig using the aad# token format required by the Speech SDK
speech_config = speechsdk.SpeechConfig(
    auth_token=f"aad#{speech_resource_id}#{token}",
    region=speech_region,
)

print("Endpoint:", speech_endpoint)
print("Region:", speech_region)
print("Token obtained:", "Yes" if token else "No")

## Step 3: Text-to-Speech -- Generate Audio from Text

We feed a paragraph of plain text to the Speech service and it produces
a `.wav` audio file using a neural voice.

In [ ]:
# Choose a neural voice
speech_config.speech_synthesis_voice_name = "en-US-JennyMultilingualNeural"

# The text we want to convert into speech
sample_text = (
    "Cloud computing has revolutionized how businesses operate. "
    "Instead of maintaining expensive on-premises servers, companies "
    "can now rent computing power on demand from providers like "
    "Microsoft Azure, scaling up or down as their needs change."
)

# Save the audio to a WAV file
output_wav = "tts_output.wav"
audio_config = speechsdk.audio.AudioConfig(filename=output_wav)

# Create the synthesizer and generate speech
synthesizer = speechsdk.SpeechSynthesizer(
    speech_config=speech_config, audio_config=audio_config
)
result = synthesizer.speak_text_async(sample_text).get()

if result.reason == speechsdk.ResultReason.SynthesizingAudioCompleted:
    print(f"Audio saved to {output_wav}")
else:
    print(f"Speech synthesis failed: {result.reason}")

## Step 4: SSML -- Fine-Grained Control over Speech

**SSML** (Speech Synthesis Markup Language) lets you add pauses, change
speaking rate, switch voices, and much more. Below we define the SSML
inline and pass it to the synthesizer.

In [ ]:
# Define SSML with pauses and emphasis
ssml_content = """
<speak version="1.0" xml:lang="en-US">
    <voice name="en-US-JennyMultilingualNeural">
        Welcome to this amazing course.
        <break strength="medium" />
        In this video, we explore how to convert text into
        natural-sounding speech using the Speech service in Foundry tools.
        <break />
        Let's get started!
    </voice>
</speak>
"""

# Save to a different file so we can compare
ssml_output_wav = "ssml_output.wav"
ssml_audio_config = speechsdk.audio.AudioConfig(filename=ssml_output_wav)

ssml_synthesizer = speechsdk.SpeechSynthesizer(
    speech_config=speech_config, audio_config=ssml_audio_config
)
ssml_result = ssml_synthesizer.speak_ssml_async(ssml_content).get()

if ssml_result.reason == speechsdk.ResultReason.SynthesizingAudioCompleted:
    print(f"SSML audio saved to {ssml_output_wav}")
else:
    print(f"SSML synthesis failed: {ssml_result.reason}")

## Step 5: Speech-to-Text -- Transcribe an Audio File

Now we do the reverse: take the audio file we generated in Step 3 and
transcribe it back into text.

> **Note:** `recognize_once_async()` transcribes the first utterance
> (up to ~15 seconds). For longer audio, use continuous recognition.

In [ ]:
# Point the recognizer at the WAV file we created earlier
speech_config.speech_recognition_language = "en-US"
audio_input = speechsdk.AudioConfig(filename="tts_output.wav")

recognizer = speechsdk.SpeechRecognizer(
    speech_config=speech_config, audio_config=audio_input
)

stt_result = recognizer.recognize_once_async().get()

if stt_result.reason == speechsdk.ResultReason.RecognizedSpeech:
    print("Transcription successful!")
    print(f"Text: {stt_result.text}")
else:
    print(f"Transcription failed: {stt_result.reason}")

# Optionally save the transcription to a file
with open("transcription.txt", "w", encoding="utf-8") as f:
    f.write(stt_result.text)
    print("Saved to transcription.txt")